# Finite-shot quantum regression: walkthrough

This optional notebook explains the checked-in experiment. The authoritative results come from the command-line runner and saved raw records. Use the project Python environment; Jupyter itself is optional and is not required by the study.

In [ ]:
from pathlib import Path
import sys
ROOT = Path.cwd()
if not (ROOT / "src").exists():
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT / "src"))
import numpy as np
from qml_project.quantum import measurement_probabilities, exact_features, sample_records, FEATURE_NAMES
from qml_project.analysis import read_metrics


## One input, exact features and sampled measurements

The same outcome record supplies singles and a pair in each of XX, YY and ZZ. A budget of 32 shots per setting costs 96 executions for one input.

In [ ]:
p = measurement_probabilities(np.array([0.5]))
counts, measured = sample_records(p, 32, np.random.default_rng(42))
for name, ideal, estimate in zip(FEATURE_NAMES, exact_features(p)[0], measured[0]):
    print(f"{name:5s} exact={ideal:+.4f} sampled={estimate:+.4f}")
print("Joint counts:", counts[0])


## Inspect measured results

Do not infer independent replication from every CSV row: measurement repetitions share the same dataset.

In [ ]:
import json
summary = json.loads((ROOT / "results/semester/summary.json").read_text())
for r in summary["summary"]:
    if r["label_noise_std"] == 0 and r["shots"] > 0:
        print(r["task"], r["shots"], r["method"], r["test_mse_mean"], r["test_mse_sd"])


## Display the primary figure

Error bars are sample SD over five dataset means after averaging acquisitions. They are not confidence intervals. Ridge does not win every condition.

In [ ]:
from IPython.display import Image, display
display(Image(filename=str(ROOT / "figures/test_mse_clean.png")))


## Reproduce

From the repository root, run `python -m pytest -q`, `python scripts/check_results.py`, and `python scripts/run_study.py --output results/my_run`. Use a fresh output directory. See README.md and docs/METHODOLOGY.md for all settings and limitations.